# Uninsured-care reimbursement analysis in SQL

This notebook uses the CDC-hosted, HRSA-published reimbursement dataset, downloaded locally with `Rscript R/download_cdc_datasets.R`, to create reusable operational metrics for testing, treatment, and vaccine payments to providers serving uninsured patients.

> The dataset is reimbursement activity, not a patient-level utilization denominator. Totals should not be presented as patient counts or rates without additional population data.

## Notebook setup

- Kernel: Python with the `duckdb` package
- SQL dialect: DuckDB
- Source: `../data/claims_reimbursement.csv` (download with `Rscript R/download_cdc_datasets.R`)
- Grain: one provider/city/state source row
- Main techniques: type cleaning, reusable views, aggregation, window functions, ranking, and data-quality checks

The SQL remains visible inside each Python cell and is executed with `con.execute(...)` or `con.sql(...)`. This avoids the `SyntaxError` that occurs when raw SQL is sent to a Python kernel.

In [1]:
import duckdb
from pathlib import Path

# DuckDB executes the SQL strings below while Python provides the notebook kernel.
con = duckdb.connect()
source_path = Path('../data/claims_reimbursement.csv')
if not source_path.exists():
    source_path = Path('data/claims_reimbursement.csv')

con.execute(f"""
-- DuckDB infers the CSV schema and preserves the original column names.
CREATE OR REPLACE TABLE uninsured_claims AS
SELECT *
FROM read_csv_auto('{source_path.as_posix()}',
    header = true,
    normalize_names = false)
""")
con.sql('SELECT COUNT(*) AS source_rows FROM uninsured_claims').df()

,source_rows
0,50244


In [9]:
# Standardize currency strings once in a staging view. Empty values
# are treated as zero only after documenting that assumption.
con.execute('''
CREATE OR REPLACE VIEW claims_clean AS
SELECT
    NULLIF(TRIM("Provider Name"), '') AS provider_name,
    UPPER(TRIM(State)) AS state,
    TRIM(City) AS city,
    COALESCE(TRY_CAST(REPLACE(REPLACE("Claims Paid for Testing", '$', ''), ',', '') AS DECIMAL(14, 2)), 0) AS testing_paid,
    COALESCE(TRY_CAST(REPLACE(REPLACE("Claims Paid for Treatment", '$', ''), ',', '') AS DECIMAL(14, 2)), 0) AS treatment_paid,
    COALESCE(TRY_CAST(REPLACE(REPLACE("Claims Paid for Vaccine", '$', ''), ',', '') AS DECIMAL(14, 2)), 0) AS vaccine_paid
FROM uninsured_claims
''')
con.sql('SELECT * FROM claims_clean LIMIT 10').df()

,provider_name,state,city,testing_paid,treatment_paid,vaccine_paid
0,ARCARE,AR,KENSENT,40.0,579.0,360.0
1,CITY OF PAGE,AZ,PAGE,0.0,1060.0,0.0
2,HOMETOWN NURSING,CA,BERKELEY,200.0,150.0,0.0
3,IMMUNO ONCOLOGY CLINIC,CA,IRVINE,1560.0,0.0,12640.0
4,HCAHEALTHONE LLC,CO,ENGLEWOOD,0.0,13388.0,0.0
5,ALACHUA PHARMACY INC,FL,ALACHUA,0.0,0.0,19137.0
6,ST CLOUD HMA PHYSICIAN,FL,SAINT CLOUD,23.0,77.0,0.0
7,AGAPEMD LLC,FL,TAVERNIER,0.0,0.0,40.0
8,CASWELL COUNTY HEALTH,GA,PROVIDENCE,0.0,6589.0,0.0
9,HUMAYUNS SC,IL,CHICAGO,0.0,6331.0,0.0


In [10]:
# State-level reimbursement mix for dashboard KPI tables.
con.execute('''
CREATE OR REPLACE VIEW state_reimbursement_summary AS
SELECT
    state,
    COUNT(*) AS provider_rows,
    SUM(testing_paid) AS testing_paid,
    SUM(treatment_paid) AS treatment_paid,
    SUM(vaccine_paid) AS vaccine_paid,
    SUM(testing_paid + treatment_paid + vaccine_paid) AS total_reimbursement,
    ROUND(100.0 * SUM(vaccine_paid) / NULLIF(SUM(testing_paid + treatment_paid + vaccine_paid), 0), 2) AS vaccine_share_pct
FROM claims_clean
GROUP BY state
''')
con.sql('''
SELECT *, DENSE_RANK() OVER (ORDER BY total_reimbursement DESC) AS reimbursement_rank
FROM state_reimbursement_summary
ORDER BY reimbursement_rank
''').df()

,state,provider_rows,testing_paid,treatment_paid,vaccine_paid,total_reimbursement,vaccine_share_pct,reimbursement_rank
0,TX,5295,1.252954e+09,1.803074e+09,99952260.0,3.155980e+09,3.17,1
1,CA,3515,1.923848e+09,3.674612e+08,184760092.0,2.476069e+09,7.46,2
2,IL,2105,1.382318e+09,1.257993e+08,320729638.0,1.828847e+09,17.54,3
3,FL,3336,5.842444e+08,5.722285e+08,106949125.0,1.263422e+09,8.47,4
4,NJ,2172,6.638960e+08,2.926359e+08,132849086.0,1.089381e+09,12.19,5
5,NC,1981,7.762597e+08,2.711267e+08,20945800.0,1.068332e+09,1.96,6
6,NY,2917,8.580092e+08,1.428309e+08,56914709.0,1.057755e+09,5.38,7
7,GA,1978,3.935408e+08,4.074976e+08,41517973.0,8.425564e+08,4.93,8
8,DC,124,6.375463e+08,7.916205e+07,3578179.0,7.202865e+08,0.50,9
9,TN,1279,3.376844e+08,1.724211e+08,8745244.0,5.188507e+08,1.69,10


In [11]:
# Provider concentration is an operational-risk metric: it shows
# whether reimbursement is dependent on a small number of providers.
con.sql('''
WITH provider_totals AS (
    SELECT provider_name, state,
           SUM(testing_paid + treatment_paid + vaccine_paid) AS total_reimbursement
    FROM claims_clean
    WHERE provider_name IS NOT NULL
    GROUP BY provider_name, state
), ranked AS (
    SELECT *,
           SUM(total_reimbursement) OVER () AS all_provider_reimbursement,
           SUM(total_reimbursement) OVER (ORDER BY total_reimbursement DESC
               ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS cumulative_reimbursement
    FROM provider_totals
)
SELECT provider_name, state, total_reimbursement,
       ROUND(100.0 * total_reimbursement / NULLIF(all_provider_reimbursement, 0), 2) AS provider_share_pct,
       ROUND(100.0 * cumulative_reimbursement / NULLIF(all_provider_reimbursement, 0), 2) AS cumulative_share_pct
FROM ranked
ORDER BY total_reimbursement DESC
LIMIT 25
''').df()

,provider_name,state,total_reimbursement,provider_share_pct,cumulative_share_pct
0,CURATIVE LABS INC,DC,646137850.0,3.44,3.44
1,FULGENT THERAPEUTICS L,CA,507386938.0,2.70,6.14
2,LABORATORY CORPORATION,NC,484273145.0,2.58,8.72
3,WALGREEN CO,IL,258705264.0,1.38,10.10
4,AEGIS SCIENCES CORPORA,TN,230669547.0,1.23,11.32
5,HELIX OPCO LLC,CA,227691475.0,1.21,12.54
6,QUEST DIAGNOSTICS INCO,MO,226616685.0,1.21,13.74
7,NOMI HEALTH INC,UT,226174599.0,1.20,14.95
8,HARRIS HEALTH SYSTEM,TX,198484573.0,1.06,16.00
9,O HARE CLINICAL LAB SE,IL,187153096.0,1.00,17.00


In [12]:
# Rows with no payment in any category deserve review. Zero may mean
# no reimbursement, a suppressed value, or an ingestion issue.
con.sql('''
SELECT state, COUNT(*) AS zero_value_rows
FROM claims_clean
WHERE testing_paid = 0 AND treatment_paid = 0 AND vaccine_paid = 0
GROUP BY state
ORDER BY zero_value_rows DESC
''').df()

,state,zero_value_rows


## Portfolio interpretation

The reusable views separate data preparation from dashboard-facing metrics. In a production model, add a calendar or reporting-period dimension, preserve source-file metadata, test currency parsing, and add a denominator before comparing reimbursement across populations.